# Canción → MIDI por instrumento en Kaggle

Notebook para uso personal. Incluye el conversor MuScriptor 0.3.0: no hace falta subir scripts.

**Antes de ejecutar:**
1. Mantén el notebook y el dataset de tu canción como **privados**.
2. En **Settings / Session options**, selecciona **Accelerator → GPU T4 x2** si está disponible (usamos una GPU) y activa **Internet**. Si Kaggle te pide verificar tu cuenta, completa ese paso.
3. En **Add Input**, añade un dataset privado con tu canción, sin incluir credenciales.
4. Acepta la licencia de [MuScriptor Medium](https://huggingface.co/MuScriptor/muscriptor-medium) con tu cuenta de Hugging Face (si ya la aceptaste no debes repetirlo).
5. Crea un token de **lectura** en [Hugging Face](https://huggingface.co/settings/tokens), con acceso al modelo autorizado. En **Add-ons → Secrets**, añádelo como **HF_TOKEN** y habilítalo para este notebook. No lo pegues en ninguna celda.

Ejecuta los bloques en orden. Se hace una prueba de 30 segundos; la canción completa requiere cambiar un interruptor más abajo. **Run All inicialmente ejecuta solo la prueba.**

La cuota se consume mientras la sesión GPU siga activa. Al terminar, descarga los ZIP y pulsa **Stop Session**. Las sesiones son temporales.


## 1. Preparar archivos y dependencias
Conservamos el PyTorch/CUDA de Kaggle en un entorno aislado. La primera instalación y descarga puede tardar varios minutos.


In [ ]:
from pathlib import Path
import os
import sys
import importlib

PROYECTO = Path("/kaggle/working/audio-a-midi")
RUNTIME_DIR = Path("/kaggle/temp/audio-midi-runtime")
PROYECTO.mkdir(parents=True, exist_ok=True)
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(RUNTIME_DIR / "huggingface")
os.environ["TORCH_HOME"] = str(RUNTIME_DIR / "torch")
os.environ["PIP_CACHE_DIR"] = str(RUNTIME_DIR / "pip")

CONVERSOR_PY = '"""Transcribe audio locally with MuScriptor and export separate MIDI tracks."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport io\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nROOT = Path(__file__).resolve().parent\n# Keep model downloads and temporary data inside the project.\nos.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))\nos.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))\n\n\nclass ConversionError(RuntimeError):\n    """An actionable error that can be shown without a traceback."""\n\n\ndef safe_name(value: str) -> str:\n    value = re.sub(r"[^\\w.-]+", "_", value, flags=re.UNICODE).strip("._")[:80]\n    return value or "instrumento"\n\n\ndef positive_float(value: str) -> float:\n    number = float(value)\n    if not math.isfinite(number) or number <= 0:\n        raise argparse.ArgumentTypeError("Debe ser un número positivo y finito.")\n    return number\n\n\ndef nonnegative_float(value: str) -> float:\n    number = float(value)\n    if not math.isfinite(number) or number < 0:\n        raise argparse.ArgumentTypeError("Debe ser un número finito mayor o igual a cero.")\n    return number\n\n\ndef decode_audio(source: Path, destination: Path, start: float, duration: float | None) -> float:\n    """Use bundled FFmpeg: MP3/WAV/FLAC/OGG/M4A -> mono PCM WAV, 16 kHz."""\n    import imageio_ffmpeg\n    import wave\n\n    command = [imageio_ffmpeg.get_ffmpeg_exe(), "-hide_banner", "-loglevel", "error", "-nostdin"]\n    if start:\n        command += ["-ss", str(start)]\n    command += ["-i", str(source), "-map", "0:a:0", "-vn"]\n    if duration is not None:\n        command += ["-t", str(duration)]\n    command += ["-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", "-y", str(destination)]\n    result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace")\n    if result.returncode:\n        raise ConversionError(f"No se pudo leer el audio. FFmpeg: {result.stderr.strip()[-2000:]}")\n    with wave.open(str(destination), "rb") as wav:\n        seconds = wav.getnframes() / wav.getframerate()\n    if seconds <= 0:\n        raise ConversionError("El archivo no contiene audio en el intervalo indicado.")\n    return seconds\n\n\ndef conductor_track(midi):\n    """Collect global metadata at absolute ticks, including later tempo changes."""\n    import mido\n\n    global_types = {"set_tempo", "time_signature", "key_signature", "marker"}\n    events = {}\n    for track in midi.tracks:\n        tick = 0\n        for message in track:\n            tick += message.time\n            if message.type in global_types:\n                key = (tick, str(message.copy(time=0)))\n                events[key] = (tick, message)\n    output = mido.MidiTrack([mido.MetaMessage("track_name", name="Tempo y compás", time=0)])\n    previous = 0\n    for tick, message in sorted(events.values(), key=lambda item: item[0]):\n        output.append(message.copy(time=tick - previous))\n        previous = tick\n    return output\n\n\ndef export_midi(midi_bytes: bytes, destination: Path) -> list[dict]:\n    """Keep one named track per predicted instrument; export solo files too.\n\n    Separate MIDI ports avoid channel collisions when there are >15 melodic\n    instruments. Solo files always use channel 1 (drums: channel 10).\n    Delta times, control changes and global tempo events are preserved.\n    """\n    import mido\n\n    midi = mido.MidiFile(file=io.BytesIO(midi_bytes))\n    if midi.type == 2:\n        raise ConversionError("Se esperaba un MIDI síncrono, pero el modelo devolvió MIDI tipo 2.")\n    tracks = [track for track in midi.tracks if any(\n        message.type == "note_on" and message.velocity > 0 for message in track\n    )]\n    if not tracks:\n        raise ConversionError("El modelo no detectó notas. Prueba otro fragmento o un modelo mayor.")\n    combined = mido.MidiFile(type=1, ticks_per_beat=midi.ticks_per_beat)\n    conductor = conductor_track(midi)\n    combined.tracks.append(conductor)\n    melodic_channels = list(range(9)) + list(range(10, 16))\n    melodic_count = 0\n    report = []\n    # Build and validate before creating any output files.\n    prepared = []\n    for index, original in enumerate(tracks, 1):\n        note_messages = [msg for msg in original if msg.type == "note_on" and msg.velocity > 0]\n        original_channels = {msg.channel for msg in note_messages}\n        if len(original_channels) != 1:\n            raise ConversionError("Una pista del modelo contiene varios canales; no puede tratarse como un solo instrumento.")\n        drums = original_channels == {9}\n        name = next((msg.name for msg in original if msg.type == "track_name"), f"Instrumento {index}")\n        programs = {msg.program for msg in original if msg.type == "program_change"}\n        if len(programs) > 1:\n            raise ConversionError(f"La pista {name!r} cambia de instrumento durante la canción.")\n        program = next(iter(programs), 0)\n        if drums:\n            port, channel = 0, 9\n        else:\n            port, channel_index = divmod(melodic_count, 15)\n            channel = melodic_channels[channel_index]\n            melodic_count += 1\n        if port > 127:\n            raise ConversionError("Demasiados instrumentos para exportar como puertos MIDI.")\n        combined_track = mido.MidiTrack([mido.MetaMessage("midi_port", port=port, time=0)])\n        solo_track = mido.MidiTrack()\n        for message in original:\n            # Replace existing port events without losing their delta times.\n            if message.type == "midi_port":\n                combined_track.append(message.copy(port=port))\n                solo_track.append(message.copy(port=0))\n            elif hasattr(message, "channel"):\n                combined_track.append(message.copy(channel=channel))\n                solo_track.append(message.copy(channel=9 if drums else 0))\n            else:\n                combined_track.append(message.copy())\n                solo_track.append(message.copy())\n        combined.tracks.append(combined_track)\n        filename = f"{index:02d}_{safe_name(name)}.mid"\n        solo = mido.MidiFile(type=1, ticks_per_beat=midi.ticks_per_beat)\n        solo.tracks.extend([mido.MidiTrack(msg.copy() for msg in conductor), solo_track])\n        prepared.append((filename, solo))\n        report.append({\n            "nombre": name, "archivo": f"instrumentos/{filename}",\n            "notas": len(note_messages), "programa_gm_0_127": program,\n            "bateria": drums, "canal_1_16": channel + 1, "puerto_0_127": port,\n        })\n    destination.mkdir(parents=True, exist_ok=True)\n    if any(destination.iterdir()):\n        raise ConversionError(f"La carpeta de salida debe estar vacía: {destination}")\n    (destination / "instrumentos").mkdir()\n    combined.save(destination / "cancion.mid")\n    for filename, solo in prepared:\n        solo.save(destination / "instrumentos" / filename)\n    return report\n\n\ndef choose_device(requested: str) -> str:\n    import torch\n\n    if requested == "auto":\n        return "cuda" if torch.cuda.is_available() else "cpu"\n    if requested == "cuda" and not torch.cuda.is_available():\n        raise ConversionError("CUDA no está disponible. Ejecuta instalar.ps1 o utiliza --dispositivo cpu.")\n    return requested\n\n\ndef diagnose() -> None:\n    import importlib.metadata\n    import imageio_ffmpeg\n    import torch\n    from huggingface_hub import get_token\n\n    print(f"Python: {sys.version.split()[0]}")\n    for package in ("muscriptor", "torch", "mido", "imageio-ffmpeg"):\n        print(f"{package}: {importlib.metadata.version(package)}")\n    print(f"FFmpeg: {imageio_ffmpeg.get_ffmpeg_exe()}")\n    print(f"CUDA disponible: {torch.cuda.is_available()}")\n    if torch.cuda.is_available():\n        print(f"GPU: {torch.cuda.get_device_name(0)}")\n    # Never show the credential itself.\n    print(f"Sesión Hugging Face configurada: {bool(get_token())}")\n    print("El acceso al modelo se comprueba al descargarlo; tener sesión no garantiza acceso.")\n\n\ndef transcribe_song(source: Path, destination: Path, *, model_size: str = "medium",\n                    device: str = "auto", start: float = 0.0,\n                    duration: float | None = None, detect_tempo: bool = False) -> dict:\n    """Reusable entry point for a future web worker. Returns the JSON report."""\n    if not math.isfinite(start) or start < 0:\n        raise ConversionError("El inicio debe ser finito y mayor o igual a cero.")\n    if duration is not None and (not math.isfinite(duration) or duration <= 0):\n        raise ConversionError("La duración debe ser positiva y finita.")\n    source, destination = Path(source).resolve(), Path(destination).resolve()\n    if not source.is_file():\n        raise ConversionError(f"No existe el archivo de audio: {source}")\n    if destination.exists() and (not destination.is_dir() or any(destination.iterdir())):\n        raise ConversionError(f"Usa una carpeta de salida nueva o vacía: {destination}")\n    if model_size not in {"small", "medium", "large"}:\n        raise ConversionError("Modelo válido: small, medium o large.")\n    if device not in {"auto", "cuda", "cpu"}:\n        raise ConversionError("Dispositivo válido: auto, cuda o cpu.")\n\n    import torch\n    from muscriptor import TranscriptionModel\n    from muscriptor.events import NoteStartEvent, ProgressEvent\n    from muscriptor.utils.download import ModelDownloadError\n\n    selected_device = choose_device(device)\n    cache = ROOT / ".cache"\n    cache.mkdir(parents=True, exist_ok=True)\n    started = time.perf_counter()\n    with tempfile.TemporaryDirectory(prefix="audio_", dir=cache) as temporary:\n        wav_path = Path(temporary) / "audio.wav"\n        seconds = decode_audio(source, wav_path, start, duration)\n        print(f"Audio: {seconds:.1f} s. Modelo: {model_size}. Dispositivo: {selected_device}.", flush=True)\n        print("Cargando modelo (la primera vez descarga los pesos)…", flush=True)\n        try:\n            model = TranscriptionModel.load_model(weights_path=model_size, device=selected_device)\n        except ModelDownloadError as error:\n            raise ConversionError(\n                f"No se pudo obtener MuScriptor {model_size}. Acepta la licencia en "\n                f"https://huggingface.co/MuScriptor/muscriptor-{model_size} "\n                "y ejecuta .\\\\autenticar.ps1. No compartas el token en el chat."\n            ) from error\n        with torch.inference_mode():\n            grid = model.detect_beat_grid_for(wav_path, "best-effort" if detect_tempo else False)\n            events = []\n            for event in model.transcribe(audio=wav_path, batch_size=1, prelude_forcing=True, use_sampling=False):\n                if isinstance(event, ProgressEvent):\n                    print(f"Progreso: {event.completed}/{event.total} bloques de audio", flush=True)\n                else:\n                    events.append(event)\n            if grid is not None:\n                grid = grid.with_onset_delay([\n                    event.start_time for event in events if isinstance(event, NoteStartEvent)\n                ])\n            midi_bytes = model.events_to_midi_bytes(iter(events), beat_grid=grid)\n        tracks = export_midi(midi_bytes, destination)\n        report = {\n            "audio": str(source), "modelo": f"MuScriptor {model_size}",\n            "dispositivo": selected_device, "inicio_audio_s": start,\n            "duracion_audio_s": round(seconds, 3), "tiempo_procesamiento_s": round(time.perf_counter() - started, 2),\n            "tempo_detectado_bpm": grid.bpm if grid is not None else None,\n            "tempo_midi_bpm": grid.bpm if grid is not None else 120,\n            "cuantizado": False, "midi": "cancion.mid", "pistas": tracks,\n        }\n        (destination / "informe.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")\n        print(f"\\nGuardado: {destination / \'cancion.mid\'}")\n        for track in tracks:\n            print(f"  {track[\'nombre\']}: {track[\'notas\']} notas -> {track[\'archivo\']}")\n        return report\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Canción a MIDI con pistas separadas por instrumento (MuScriptor local).")\n    parser.add_argument("audio", nargs="?", type=Path, help="Archivo MP3, WAV, FLAC, OGG, M4A, etc.")\n    parser.add_argument("--salida", type=Path, help="Carpeta nueva o vacía para los resultados.")\n    parser.add_argument("--modelo", choices=["small", "medium", "large"], default="medium")\n    parser.add_argument("--dispositivo", choices=["auto", "cuda", "cpu"], default="auto")\n    parser.add_argument("--inicio", type=nonnegative_float, default=0.0, help="Inicio del fragmento, en segundos.")\n    parser.add_argument("--duracion", type=positive_float, help="Transcribir solo estos segundos; por defecto, hasta el final.")\n    parser.add_argument("--detectar-tempo", action="store_true", help="Detectar BPM/compás (descarga un modelo adicional).")\n    parser.add_argument("--diagnostico", action="store_true", help="Comprobar instalación, GPU y sesión sin descargar pesos.")\n    args = parser.parse_args(argv)\n    if not args.audio and not args.diagnostico:\n        parser.error("Indica un archivo de audio o utiliza --diagnostico.")\n    try:\n        if args.diagnostico:\n            diagnose()\n        if args.audio:\n            destination = args.salida or ROOT / "resultados" / (safe_name(args.audio.stem) + "_" + time.strftime("%Y%m%d_%H%M%S"))\n            transcribe_song(args.audio, destination, model_size=args.modelo, device=args.dispositivo,\n                            start=args.inicio, duration=args.duracion, detect_tempo=args.detectar_tempo)\n        return 0\n    except ImportError as error:\n        print(f"Falta una dependencia: {error}. Ejecuta instalar.ps1.", file=sys.stderr)\n    except ConversionError as error:\n        print(f"Error: {error}", file=sys.stderr)\n    except KeyboardInterrupt:\n        print("Conversión interrumpida.", file=sys.stderr)\n        return 130\n    except Exception as error:\n        if "out of memory" in str(error).lower():\n            print("Memoria GPU insuficiente. Prueba --modelo small o --dispositivo cpu.", file=sys.stderr)\n        else:\n            print(f"Error ({type(error).__name__}): {error}", file=sys.stderr)\n        return 1\n    return 1\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
RUNTIME_PY = '"""Kaggle setup and result packaging; credentials stay in process memory."""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport uuid\nimport zipfile\n\n\nGPU_CHECK = \'\'\'\nimport json\nimport torch\nif not torch.cuda.is_available():\n    raise SystemExit("GPU no disponible. Activa Accelerator > GPU en Kaggle y reinicia la sesión.")\ntry:\n    x = torch.ones((32, 32), device="cuda")\n    result = (x @ x).sum().item()\n    torch.cuda.synchronize()\nexcept Exception:\n    raise SystemExit("PyTorch no puede calcular con esta GPU. Prueba GPU T4 x2 y reinicia la sesión.") from None\nif result != 32768.0:\n    raise SystemExit("La comprobación de cálculo CUDA falló.")\nprint(json.dumps({"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),\n                  "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)}))\n\'\'\'\n\n\ndef run_checked(command: list[str], *, env: dict | None = None, cwd: Path | None = None) -> None:\n    """Stream progress without placing tokens in command lines or exceptions."""\n    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                               text=True, encoding="utf-8", errors="replace", env=env, cwd=cwd)\n    try:\n        assert process.stdout is not None\n        for line in process.stdout:\n            print(line, end="", flush=True)\n        code = process.wait()\n    except BaseException:\n        process.terminate()\n        try:\n            process.wait(timeout=10)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n        raise\n    finally:\n        if process.stdout is not None:\n            process.stdout.close()\n    if code:\n        raise RuntimeError(f"El proceso terminó con código {code}. Revisa el mensaje anterior.")\n\n\ndef prepare_runtime(runtime_dir: Path) -> Path:\n    """Reuse Kaggle\'s CUDA PyTorch in an isolated environment, without upgrading it."""\n    if sys.version_info < (3, 10):\n        raise RuntimeError("MuScriptor necesita Python 3.10 o posterior.")\n    print("Comprobando la GPU antes de instalar o descargar modelos…", flush=True)\n    run_checked([sys.executable, "-c", GPU_CHECK])\n    versions = json.loads(subprocess.check_output([\n        sys.executable, "-c",\n        "import importlib.metadata as m, json; "\n        "print(json.dumps({p:m.version(p) for p in (\'torch\',\'torchaudio\',\'numpy\')}))",\n    ], text=True))\n    runtime_dir = Path(runtime_dir)\n    runtime_dir.mkdir(parents=True, exist_ok=True)\n    environment = runtime_dir / "venv"\n    python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")\n    if not python.is_file():\n        run_checked([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", str(environment)])\n    constraints = runtime_dir / "constraints.txt"\n    constraints.write_text("".join(f"{key}=={value}\\n" for key, value in versions.items()), encoding="utf-8")\n    run_checked([\n        sys.executable, "-m", "pip", "--python", str(python), "install", "--disable-pip-version-check",\n        "--constraint", str(constraints), "muscriptor==0.3.0", "mido>=1.3,<2",\n        "imageio-ffmpeg>=0.6,<0.7", "huggingface-hub==2.2.0",\n    ])\n    run_checked([str(python), "-c", GPU_CHECK])\n    run_checked([str(python), "-c", "import muscriptor, torchaudio, mido, imageio_ffmpeg; print(\'Dependencias listas.\')"])\n    return python\n\n\ndef find_audio(input_root: Path, selected: str = "") -> Path:\n    """Auto-select a single audio file; require a choice when several exist."""\n    if selected:\n        source = Path(selected).expanduser()\n        if not source.is_absolute():\n            source = Path(input_root) / source\n        if not source.is_file():\n            raise ValueError(f"No existe el audio: {source}")\n        return source.resolve()\n    extensions = {".mp3", ".wav", ".flac", ".ogg", ".m4a", ".aac", ".aiff", ".aif", ".opus"}\n    files = sorted(path for path in Path(input_root).rglob("*") if path.is_file() and path.suffix.lower() in extensions)\n    if not files:\n        raise ValueError("No hay canciones en /kaggle/input. Añade un dataset privado con tu audio mediante Add Input.")\n    if len(files) != 1:\n        candidates = "\\n".join(str(path) for path in files[:30])\n        raise ValueError("Hay varias canciones. Copia una ruta en ARCHIVO_AUDIO:\\n" + candidates)\n    return files[0].resolve()\n\n\ndef conversion_command(python: Path, project: Path, source: Path, destination: Path, *,\n                       model: str = "medium", start: float = 0.0, duration: float | None = None) -> list[str]:\n    command = [str(python), "-u", str(Path(project) / "audio_a_midi.py"), str(source),\n               "--salida", str(destination), "--modelo", model, "--dispositivo", "cuda",\n               "--inicio", str(start)]\n    if duration is not None:\n        command.extend(["--duracion", str(duration)])\n    return command\n\n\ndef new_destination(project: Path, source: Path, label: str) -> Path:\n    from audio_a_midi import safe_name\n    return Path(project) / "resultados" / f"{safe_name(Path(source).stem)}_{label}_{uuid.uuid4().hex[:10]}"\n\n\ndef package_result(destination: Path) -> Path:\n    """Validate MIDI and zip an explicit allowlist; never include audio or caches."""\n    import mido\n\n    destination = Path(destination).resolve()\n    report = json.loads((destination / "informe.json").read_text(encoding="utf-8"))\n    combined = mido.MidiFile(destination / "cancion.mid")\n    if combined.type != 1 or not report["pistas"]:\n        raise ValueError("El resultado no contiene un MIDI tipo 1 con instrumentos.")\n    files = [destination / "cancion.mid", destination / "informe.json"]\n    for track in report["pistas"]:\n        path = (destination / track["archivo"]).resolve()\n        if not path.is_relative_to(destination / "instrumentos") or path.suffix != ".mid":\n            raise ValueError("El informe contiene una ruta de instrumento no válida.")\n        solo = mido.MidiFile(path)\n        starts = [msg for tr in solo.tracks for msg in tr if msg.type == "note_on" and msg.velocity > 0]\n        if len(starts) != track["notas"]:\n            raise ValueError(f"Número de notas incorrecto en {path.name}.")\n        expected_channel = 9 if track["bateria"] else 0\n        if any(msg.channel != expected_channel for msg in starts):\n            raise ValueError(f"Canal incorrecto en {path.name}.")\n        files.append(path)\n    readme = destination / "LEEME.txt"\n    readme.write_text(\n        "Canción a MIDI — MuScriptor 0.3.0\\n\\n"\n        "cancion.mid: todas las pistas. instrumentos/: un MIDI por instrumento.\\n"\n        "Importa los archivos desde el mismo origen temporal. Batería: canal MIDI 10.\\n"\n        "120 BPM es una referencia técnica para mantener el tiempo original; no es una medición del tempo.\\n"\n        "Las notas no están cuantizadas. Los instrumentos y las notas son predicciones.\\n"\n        "MuScriptor, Kyutai y Mirelo: https://github.com/muscriptor/muscriptor\\n"\n        "Pesos CC BY-NC 4.0: uso no comercial.\\n", encoding="utf-8")\n    files.append(readme)\n    archive = Path(str(destination) + ".zip")\n    with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as output:\n        for path in files:\n            output.write(path, arcname=path.relative_to(destination).as_posix())\n    return archive\n'

(PROYECTO / "audio_a_midi.py").write_text(CONVERSOR_PY, encoding="utf-8")
(PROYECTO / "kaggle_runtime.py").write_text(RUNTIME_PY, encoding="utf-8")
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))
import kaggle_runtime
importlib.reload(kaggle_runtime)
PYTHON_MIDI = kaggle_runtime.prepare_runtime(RUNTIME_DIR)


## 2. Conectar Hugging Face
Leemos HF_TOKEN desde Secrets; el token no se guarda en archivos ni se imprime. La sesión local de tu PC no se transfiere a Kaggle.


In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    token_hf = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("Añade HF_TOKEN en Add-ons > Secrets y habilítalo para este notebook.") from None
if not token_hf or not token_hf.strip():
    raise RuntimeError("HF_TOKEN está vacío. Actualízalo en Secrets.")
ENTORNO_MIDI = os.environ.copy()
ENTORNO_MIDI["HF_TOKEN"] = token_hf.strip()
del token_hf
kaggle_runtime.run_checked([str(PYTHON_MIDI), "-c", "from huggingface_hub import HfApi; HfApi().whoami(); print('Conexión con Hugging Face correcta.')"], env=ENTORNO_MIDI)


## 3. Elegir la canción
Con una sola canción en los datos de entrada no debes cambiar nada. Si hay varias, especifica la ruta completa de Kaggle en ARCHIVO_AUDIO (no la ruta C:\ de tu PC).


In [ ]:
# Deja vacío para detectar automáticamente una única canción.
ARCHIVO_AUDIO = ""
MODELO = "medium"
INICIO_SEGUNDOS = 0.0

if MODELO not in {"small", "medium", "large"}:
    raise ValueError("Modelo válido: small, medium o large.")
# Small y Large requieren aceptar sus propias licencias antes de usarlos.
AUDIO = kaggle_runtime.find_audio(Path("/kaggle/input"), ARCHIVO_AUDIO)
print("Canción seleccionada:", AUDIO.name)


## 4. Probar 30 segundos
Se muestran los bloques procesados. El ZIP aparecerá como enlace y también en los archivos de salida.


In [ ]:
from IPython.display import FileLink, display

def convertir_y_descargar(etiqueta, duracion):
    destino = kaggle_runtime.new_destination(PROYECTO, AUDIO, etiqueta)
    comando = kaggle_runtime.conversion_command(
        PYTHON_MIDI, PROYECTO, AUDIO, destino,
        model=MODELO, start=INICIO_SEGUNDOS, duration=duracion,
    )
    kaggle_runtime.run_checked(comando, env=ENTORNO_MIDI, cwd=PROYECTO)
    # Empaquetamos con el mismo Python que tiene mido instalado.
    empaquetar = "from pathlib import Path; from kaggle_runtime import package_result; import sys; package_result(Path(sys.argv[1]))"
    kaggle_runtime.run_checked([str(PYTHON_MIDI), "-c", empaquetar, str(destino)], cwd=PROYECTO)
    archivo_zip = Path(str(destino) + ".zip")
    print("ZIP listo:", archivo_zip.name)
    display(FileLink(str(archivo_zip.relative_to(Path.cwd()))))
    print("Si el enlace no descarga, busca el ZIP en Output / Files > audio-a-midi > resultados.")
    return archivo_zip

ZIP_PRUEBA = convertir_y_descargar("prueba_30s", 30.0)


## 5. Convertir la canción completa
Tras revisar la prueba, cambia CONVERTIR_COMPLETA a True y ejecuta **solo este bloque**. Repetir un bloque crea otra carpeta para conservar los resultados anteriores.


In [ ]:
CONVERTIR_COMPLETA = False

if CONVERTIR_COMPLETA:
    ZIP_COMPLETO = convertir_y_descargar("completa", None)
else:
    print("Conversión completa desactivada. Cambia CONVERTIR_COMPLETA a True para ejecutarla.")


## Al terminar
- Descarga el ZIP antes de detener la sesión. Contiene cancion.mid, instrumentos/*.mid, informe.json y LEEME.txt.
- Guarda el notebook con **Quick Save** si solo quieres conservar el código. **Save & Run All** vuelve a ejecutar las celdas y consume GPU; también ejecutará la canción completa si dejaste el interruptor en True.
- Pulsa **Stop Session**; cerrar la pestaña no es suficiente para ahorrar cuota.

**Interpretar el MIDI:** 120 BPM es una referencia técnica, no el tempo detectado de la canción. Se conservan los tiempos de las notas sin cuantizar. Los instrumentos y notas son predicciones; dos intérpretes del mismo instrumento pueden compartir pista. No se generan pistas de audio aisladas.

**Errores habituales:**
- GPU ausente: activa el acelerador y reinicia la sesión. Fallo de cálculo CUDA con P100: selecciona T4 si está disponible.
- Instalación fallida: comprueba Internet; no fuerces actualizar PyTorch/CUDA sin revisar el error.
- Acceso denegado al modelo: comprueba el token de lectura y la licencia en la misma cuenta de Hugging Face.
- Varias canciones: copia una de las rutas mostradas en ARCHIVO_AUDIO.
- Sin memoria: prueba Medium o Small; Small requiere aceptar su licencia.
- Sesión nueva: vuelve a ejecutar la preparación y conexión; las dependencias y cachés temporales pueden desaparecer.

[MuScriptor — Kyutai y Mirelo](https://github.com/muscriptor/muscriptor). Código MIT; pesos CC BY-NC 4.0, uso no comercial.
[Documentación de Kaggle](https://www.kaggle.com/docs/notebooks) · [Uso eficiente de GPU](https://www.kaggle.com/docs/efficient-gpu-usage).
